# Soluzioni — Lezione 3: Imparare dagli esempi
[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/main/ml-humanities/solutions/03_imparare_dagli_esempi_svm_soluzioni.ipynb)

In [1]:
# ⚙️ CELLA DI PREPARAZIONE — eseguitela per prima (Shift+Invio) e non preoccupatevi del codice.
# Cerca la cartella "data" del corso; se non la trova (per esempio su Google Colab),
# scarica il materiale del corso da GitHub. Funziona anche senza internet, se avete la cartella.
from pathlib import Path
import subprocess

def trova_dati():
    for p in [Path("../data"), Path("data"), Path("ml-humanities/data"), Path("yuribizzoni/ml-humanities/data")]:
        if (p / "lessico_felicita.csv").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuri-bizzoni/yuribizzoni"], check=True)
    return Path("yuribizzoni/ml-humanities/data")

DATI = trova_dati()
print("Dati trovati in:", DATI.resolve())

Dati trovati in: /home/user/yuribizzoni/ml-humanities/data


In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

passaggi = pd.read_csv(DATI / "passaggi_autori.csv")
X_train, X_test, y_train, y_test = train_test_split(
    passaggi["testo"], passaggi["autore"], test_size=0.25, random_state=0, stratify=passaggi["autore"])
print(len(X_train), "brani per addestrare,", len(X_test), "per verificare")

450 brani per addestrare, 150 per verificare


In [3]:
modello = make_pipeline(TfidfVectorizer(), LinearSVC()).fit(X_train, y_train)
print(f"Modello di base (parole): {modello.score(X_test, y_test):.1%}")

Modello di base (parole): 100.0%


## Esercizio 1 — Mettere alla prova il modello

In [4]:
mie = [
    "— Babbo mio, perdonatemi! — disse il burattino piangendo, e la Fata lo guardò con tenerezza.",
    "Renzo, che aveva nel cuore la sua Lucia, camminava pensando alla Provvidenza e al curato.",
    "Il mio caso, signori, è curioso: io rido, e intanto mi vedo ridere, e non mi riconosco più.",
    "Che felicità oggi!!! Finalmente vacanza, ci vediamo tutti al mare 😊",
]
for frase, autore in zip(mie, modello.predict(mie)):
    print(f"{autore:>11}  ←  {frase}")

    Collodi  ←  — Babbo mio, perdonatemi! — disse il burattino piangendo, e la Fata lo guardò con tenerezza.
    Manzoni  ←  Renzo, che aveva nel cuore la sua Lucia, camminava pensando alla Provvidenza e al curato.
    Collodi  ←  Il mio caso, signori, è curioso: io rido, e intanto mi vedo ridere, e non mi riconosco più.
 Pirandello  ←  Che felicità oggi!!! Finalmente vacanza, ci vediamo tutti al mare 😊


**Commento.** Un classificatore **sceglie sempre** tra le classi che conosce: non ha l'opzione "nessuno dei cinque".
Anche un messaggio del 2026 viene assegnato a uno scrittore dell'Ottocento, in base alle poche parole che riconosce.
Un modello è valido solo dentro il mondo dei suoi dati di addestramento.

## Esercizio 2 — Lettere invece di parole

In [5]:
car = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)), LinearSVC())
car.fit(X_train, y_train)
print(f"Parole:    {modello.score(X_test, y_test):.1%}")
print(f"Caratteri: {car.score(X_test, y_test):.1%}")

Parole:    100.0%
Caratteri: 98.7%


**Commento.** Qui entrambi i metodi sono quasi perfetti: con un libro per autore il compito è troppo facile per vedere differenze
(lo discuteremo nella lezione 4). In generale le sequenze di caratteri catturano **desinenze, grafie, apostrofi, punteggiatura**
(per esempio `— ` o `;`): tratti di stile che le parole intere perdono, e funzionano bene nell'attribuzione d'autore con testi brevi.

## Esercizio 3 — Senza nomi propri

In [6]:
nomi = ["pinocchio", "geppetto", "burattino", "fata", "lucignolo", "renzo", "lucia", "agnese", "abbondio", "rodrigo",
        "perpetua", "cristoforo", "innominato", "gertrude", "federigo", "borromeo", "tonio", "griso", "ferrer",
        "giovanni", "giovanna", "brontu", "isidoro", "costantino", "porru", "predu", "malafede", "zia", "bera",
        "nanni", "tota", "concetta", "nunziata", "maria", "elisa", "rosa"]
# aggiungiamo i nomi più "pesanti" trovati dal modello di base (parole con iniziale maiuscola nei testi)
maiuscole = pd.Series(re.findall(r"(?<![.!?»—] )\b[A-Z][a-zà-ù]+", " ".join(passaggi["testo"]))).str.lower().value_counts()
nomi = sorted(set(nomi) | set(maiuscole[maiuscole >= 8].index))
print(len(nomi), "parole escluse, per esempio:", nomi[:15])

senza_nomi = make_pipeline(TfidfVectorizer(stop_words=nomi), LinearSVC()).fit(X_train, y_train)
print(f"Con i nomi:   {modello.score(X_test, y_test):.1%}")
print(f"Senza nomi:   {senza_nomi.score(X_test, y_test):.1%}")

120 parole escluse, per esempio: ['abbecedario', 'abbondio', 'agnese', 'amante', 'anna', 'arlecchino', 'bachisia', 'bax', 'befana', 'bellavita', 'bera', 'borromeo', 'brontu', 'burattino', 'carlo']


Con i nomi:   100.0%
Senza nomi:   100.0%


**Commento.** L'accuratezza non cambia: il modello non si affida solo ai nomi, ma a migliaia di piccoli indizi lessicali e
grafici distribuiti in tutto il testo. È una buona notizia (il modello non è solo un "trova-nomi") e una cattiva notizia
(i suoi indizi sono difficili da riassumere in una regola che un essere umano possa discutere). Nota: la lista di "maiuscole"
include anche parole a inizio frase: è un filtro grezzo, da migliorare.